# Data Visualization: Marital Status vs Qualification
**Assignment – Data Science Visualization**  
This notebook visualizes the relationship between **marital status** and **educational qualification** using Matplotlib, Seaborn, and Plotly.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## 2. Generate Sample Dataset

In [ ]:
np.random.seed(42)

qualifications = ['Matric', 'Intermediate', 'Bachelor', 'Master', 'PhD']
marital_statuses = ['Single', 'Married', 'Divorced', 'Widowed']

# Probabilities of marital status per qualification level
probs = {
    'Matric':       [0.45, 0.35, 0.12, 0.08],
    'Intermediate': [0.40, 0.40, 0.12, 0.08],
    'Bachelor':     [0.38, 0.46, 0.10, 0.06],
    'Master':       [0.28, 0.55, 0.11, 0.06],
    'PhD':          [0.20, 0.60, 0.13, 0.07],
}

records = []
sizes = {'Matric': 250, 'Intermediate': 420, 'Bachelor': 860, 'Master': 620, 'PhD': 230}

for qual in qualifications:
    n = sizes[qual]
    chosen = np.random.choice(marital_statuses, size=n, p=probs[qual])
    for status in chosen:
        records.append({'Qualification': qual, 'Marital_Status': status})

df = pd.DataFrame(records)
df['Qualification'] = pd.Categorical(df['Qualification'], categories=qualifications, ordered=True)

print(f'Dataset shape: {df.shape}')
print(f'\nFirst 5 rows:')
df.head()

## 3. Basic Data Exploration

In [ ]:
print('--- Value Counts: Qualification ---')
print(df['Qualification'].value_counts().sort_index())
print('\n--- Value Counts: Marital Status ---')
print(df['Marital_Status'].value_counts())
print('\n--- Cross Tabulation ---')
crosstab = pd.crosstab(df['Qualification'], df['Marital_Status'])
print(crosstab)

## 4. Visualization 1 – Stacked Bar Chart (Matplotlib)

In [ ]:
crosstab = pd.crosstab(df['Qualification'], df['Marital_Status'])
crosstab = crosstab.reindex(qualifications)

colors = ['#534AB7', '#1D9E75', '#D85A30', '#888780']
statuses = crosstab.columns.tolist()

fig, ax = plt.subplots(figsize=(10, 6))

bottom = np.zeros(len(qualifications))
for i, status in enumerate(statuses):
    bars = ax.bar(qualifications, crosstab[status], bottom=bottom,
                  color=colors[i], label=status, edgecolor='white', linewidth=0.5)
    # Add count labels inside bars
    for bar, val in zip(bars, crosstab[status]):
        if val > 20:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_y() + bar.get_height()/2,
                    str(val), ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')
    bottom += crosstab[status].values

ax.set_title('Marital Status Distribution by Qualification Level', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Qualification', fontsize=12)
ax.set_ylabel('Number of People', fontsize=12)
ax.legend(title='Marital Status', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('stacked_bar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Stacked bar chart saved as stacked_bar_chart.png')

## 5. Visualization 2 – Grouped Bar Chart (Seaborn)

In [ ]:
count_df = df.groupby(['Qualification', 'Marital_Status']).size().reset_index(name='Count')

fig, ax = plt.subplots(figsize=(12, 6))

sns.barplot(
    data=count_df,
    x='Qualification', y='Count', hue='Marital_Status',
    palette=['#534AB7', '#1D9E75', '#D85A30', '#888780'],
    order=qualifications,
    ax=ax
)

ax.set_title('Marital Status Count by Qualification (Grouped)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Qualification', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.legend(title='Marital Status', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('grouped_bar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grouped bar chart saved as grouped_bar_chart.png')

## 6. Visualization 3 – Heatmap (Seaborn)

In [ ]:
# Normalize crosstab to percentages
crosstab_pct = crosstab.div(crosstab.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 5))

sns.heatmap(
    crosstab_pct, annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Percentage (%)'}, ax=ax
)

ax.set_title('Heatmap: Marital Status % by Qualification', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Marital Status', fontsize=12)
ax.set_ylabel('Qualification', fontsize=12)
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved as heatmap.png')

## 7. Visualization 4 – Interactive Chart (Plotly)

In [ ]:
count_df = df.groupby(['Qualification', 'Marital_Status']).size().reset_index(name='Count')
count_df['Qualification'] = pd.Categorical(count_df['Qualification'], categories=qualifications, ordered=True)
count_df = count_df.sort_values('Qualification')

fig = px.bar(
    count_df,
    x='Qualification', y='Count', color='Marital_Status',
    barmode='stack',
    color_discrete_map={
        'Single': '#534AB7',
        'Married': '#1D9E75',
        'Divorced': '#D85A30',
        'Widowed': '#888780'
    },
    title='Marital Status Distribution by Qualification (Interactive)',
    labels={'Count': 'Number of People', 'Qualification': 'Qualification Level'},
    template='plotly_white'
)

fig.update_layout(
    legend_title_text='Marital Status',
    title_font_size=16,
    bargap=0.2
)

fig.write_html('interactive_chart.html')
fig.show()
print('Interactive chart saved as interactive_chart.html')

## 8. Summary & Observations

In [ ]:
print('=== KEY OBSERVATIONS ===')
print()
total = len(df)
for status in marital_statuses:
    count = (df['Marital_Status'] == status).sum()
    print(f'{status:10s}: {count:4d} ({count/total*100:.1f}%)')

print()
print('--- Most common marital status per qualification ---')
for qual in qualifications:
    subset = df[df['Qualification'] == qual]
    most_common = subset['Marital_Status'].value_counts().idxmax()
    print(f'{qual:15s} → {most_common}')